### ChatBot Evaluation

In [16]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")
os.environ["GEMINI_API_KEY"]=os.getenv("GEMINI_API_KEY")
os.environ["LANGSMITH_TRACING"]="true"

In [17]:
from langsmith import Client
client = Client()

dataset_name = "Simple Chatbot Evaluation"
dataset = client.create_dataset(dataset_name)

client.create_examples(
    dataset_id = dataset.id,
    examples = [
    {
        "inputs": {"question": "What is Python?"},
        "outputs": {
            "answer": "Python is a high-level programming language."
        }
    },
    {
        "inputs": {"question": "What is FastAPI?"},
        "outputs": {
            "answer": "FastAPI is a Python framework for building APIs."
        }
    },
    {
        "inputs": {"question": "What is MongoDB?"},
        "outputs": {
            "answer": "MongoDB is a NoSQL database that stores data in documents."
        }
    },
    {
        "inputs": {"question": "What is artificial intelligence?"},
        "outputs": {
            "answer": "Artificial intelligence is the ability of machines to perform tasks that normally require human intelligence."
        }
    },
    {
        "inputs": {"question": "What is machine learning?"},
        "outputs": {
            "answer": "Machine learning is a branch of AI where computers learn patterns from data."
        }
    }
]
)

{'example_ids': ['8aede651-56bc-4dfd-be86-11af74f096f0',
  'dd83c680-b1c1-4c0e-af2e-4ac503c90ab5',
  'bb062df7-0a3d-4e12-a4f2-47a4856ab64c',
  'dd7dc5d8-f57b-4466-8630-7eb6429fb563',
  'a1e49296-45bd-4fd9-8f37-040dd6b5e066'],
 'count': 5,
 'as_of': '2026-10-08T21:01:06.375449476Z'}

### Define Metrics (LLM as a Judge)

In [23]:
from google.genai import Client
from google.genai import types
from langsmith import wrappers

gemini_client = wrappers.wrap_gemini(Client())

eval_instruction = """
You are an expert professor specialized in grading students' answers to questions.
"""

def correctness(inputs: dict,outputs: dict,reference_outputs: dict) -> bool:

    user_content = f"""
        Question:
        {inputs["question"]}
        Here is the real answer:
        {reference_outputs["answer"]}
        You are grading the following predicted answer:
        {outputs["response"]}
        Respond with ONLY:CORRECT or INCORRECT
        Grade:
    """

    response = gemini_client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=user_content,
        config=types.GenerateContentConfig(
            system_instruction=eval_instruction,
            temperature=0
        )
    )

    return response.text.strip() == "CORRECT"


In [32]:
def concision(outputs: dict,reference_outputs: dict) -> bool:

    predicted = outputs.get("response", "")
    reference = reference_outputs.get("answer", "")

    if not predicted:
        return False

    return len(predicted) < 2 * len(reference)

### Run Evaluation    

In [33]:
default_instruction = "Respond to the users question in a short, concise manner (one short sentence)"
def my_app(question: str, model:str = "gemini-3.5-flash-lite", instructions:str = default_instruction) ->str:
    return gemini_client.models.generate_content(    
        model=model,
        contents = question,
        config={
            "system_instruction":instructions,
            "temperature":0
        }
        ).text

In [34]:
def is_target(inputs:str) -> dict:
    return {"response": my_app(inputs["question"])}

In [35]:
experiment_result = client.evaluate(
    is_target,
    data = dataset_name,
    evaluators= [correctness,concision],
    experiment_prefix="gemini-3.5-flash-lite-chatbot"
)

View the evaluation results for experiment: 'gemini-3.5-flash-lite-chatbot-e23aebc1' at:
https://smith.langchain.com/o/c0b0414e-ffe5-41d7-88c6-d5169f77e0bd/datasets/127c7c60-4ba1-4c96-adbc-2eeb02e35276/compare?selectedSessions=b48640c6-bee1-49f6-a2e7-21aa5cec782d


